# Contrastive Pair Export & Pilot Validation

This notebook joins the prediction log with the cached BridgeData V2 manifest,
runs the pilot validation of the directional-consistency metric against
early-motion ground truth, and exports `pairs.json` for the
external Isaac Sim visualiser.

The model is never loaded here; inputs are the CSVs produced by the previous
notebooks.

**Prerequisite:** predictions must have been logged with `pair_id`,
`role` (`'a'`/`'b'`) and `scene_id` passed via `**extra` in
`append_prediction_log`, with `scene_id` matching `episode_index` in the
manifest.


## 1. Mount Drive


In [8]:
from google.colab import drive
drive.mount('/content/drive')
import os
PRED_CSV = '/content/drive/MyDrive/openvla_cache/probe_predictions.csv'
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/bridge_multiobj'
OUT_JSON  = '/content/drive/MyDrive/openvla_cache/pairs.json'
MANIFEST_CSV = os.path.join(CACHE_DIR, 'manifest.csv')
print('predictions ->', PRED_CSV)
print('manifest    ->', MANIFEST_CSV)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
predictions -> /content/drive/MyDrive/openvla_cache/probe_predictions.csv
manifest    -> /content/drive/MyDrive/openvla_cache/bridge_multiobj/manifest.csv


## 2. Get the code and import `export_pairs.py`

Clones the repo fresh and clears the import cache, mirroring the pattern used
for `model.py` and `data.py`.


In [9]:
import sys, shutil, os, glob, importlib
REPO_DIR = '/content/ECS8056'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q https://github.com/LewisTL/ECS8056.git {REPO_DIR}

module_dir = os.path.dirname(glob.glob(os.path.join(REPO_DIR, '**', 'export_pairs.py'), recursive=True)[0])
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
sys.modules.pop('export_pairs', None)
importlib.invalidate_caches()

import export_pairs
from export_pairs import load_inputs, build_pairs, write_pairs
print('imported export_pairs.py from', module_dir)
print('BRIDGE_TO_ISAAC =\n', export_pairs.BRIDGE_TO_ISAAC)

imported export_pairs.py from /content/ECS8056
BRIDGE_TO_ISAAC =
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


## 3. Load the prediction log and manifest

`load_inputs` validates that the prediction log carries the required probe
columns and fails loudly if any are missing.


In [10]:
preds, manifest = load_inputs(PRED_CSV, MANIFEST_CSV)
print(f'{len(preds)} logged predictions | {preds["pair_id"].nunique()} pair ids | '
      f'{preds["scene_id"].nunique()} scenes')
print(f'{len(manifest)} manifest rows')
preds.head(4)

158 logged predictions | 79 pair ids | 72 scenes
224 manifest rows


,timestamp,instruction,unnorm_key,do_sample,gpu_name,gpu_capability,dtype,seed,torch,transformers,...,a2,a3,a4,a5,a6,scene_id,pair_id,role,spatial_term,sample_idx
0,2026-07-12T23:33:54.020611+00:00,Place the can to the left of the pot.,bridge_orig,False,NVIDIA A100-SXM4-80GB,sm_80,bfloat16,42,2.11.0+cu128,4.40.1,...,-0.008036,0.001138,0.025344,0.007992,0.996078,0,ep000000_left,a,left,0
1,2026-07-12T23:33:54.790916+00:00,Place the can to the right of the pot.,bridge_orig,False,NVIDIA A100-SXM4-80GB,sm_80,bfloat16,42,2.11.0+cu128,4.40.1,...,-0.008815,0.001138,0.030033,-0.009737,0.996078,0,ep000000_left,b,left,0
2,2026-07-12T23:33:55.568129+00:00,Slide the cloth diagonally to the front of the...,bridge_orig,False,NVIDIA A100-SXM4-80GB,sm_80,bfloat16,42,2.11.0+cu128,4.40.1,...,-0.023366,-0.041009,0.022665,-0.008126,0.996078,2,ep000002_front,a,front,0
3,2026-07-12T23:33:56.330973+00:00,Slide the cloth diagonally to the back of the ...,bridge_orig,False,NVIDIA A100-SXM4-80GB,sm_80,bfloat16,42,2.11.0+cu128,4.40.1,...,-0.018689,-0.034623,0.024674,0.001545,0.996078,2,ep000002_front,b,front,0


## 4. Build contrastive pairs

Predictions are grouped by `pair_id`; repeated predictions per role
(samples or paraphrases) are averaged, retaining per-axis standard deviation
and count for the effect-size analysis. Pair ids lacking both roles are
reported and skipped.


In [11]:
pairs, skipped = build_pairs(preds, manifest)
with_gt = sum(1 for p in pairs if 'gt_vector' in p)
print(f'{len(pairs)} complete pairs | {skipped} incomplete pair ids skipped | '
      f'{with_gt} pairs joined to ground truth')

79 complete pairs | 0 incomplete pair ids skipped | 79 pairs joined to ground truth


## 5. Pilot validation - sign agreement against ground truth

Gate for the directional metric and the frame convention (plan: metrics are
validated against trajectories whose ground-truth motion direction is known).
For each prediction, the sign of the predicted translation on the dominant
ground-truth axis is compared with the ground-truth sign.

Interpretation:
* **Near-zero agreement on one axis** indicates a flipped axis between the
  Bridge action frame and the assumed frame, set the corresponding row of
  `BRIDGE_TO_ISAAC` in `export_pairs.py` and rebuild.
* **Near-chance agreement on all axes** means the check carries no signal on
  this subset; restrict to scenes with large, unambiguous ground-truth motion
  before concluding anything about the frame.
* Model failure is expected on *some* scenes so the gate is systematic, 
  axis-level disagreement, not per-scene misses.


In [12]:
import numpy as np
import pandas as pd

MIN_GT_NORM = 0.01   # exclude near-static ground truth; tune against the data

rows = []
for p in pairs:
    gt = p.get('gt_vector')
    if gt is None:
        continue
    gt_t = np.asarray(gt[:3])
    if np.linalg.norm(gt_t) < MIN_GT_NORM:
        continue
    dom = int(np.argmax(np.abs(gt_t)))
    for role in ('a', 'b'):
        act_t = np.asarray(p[f'action_{role}'][:3])
        rows.append({
            'pair_id': p['pair_id'],
            'role': role,
            'dominant_axis': 'xyz'[dom],
            'gt_sign': float(np.sign(gt_t[dom])),
            'pred_sign': float(np.sign(act_t[dom])),
            'agree': bool(np.sign(act_t[dom]) == np.sign(gt_t[dom])),
        })

pilot = pd.DataFrame(rows)
print(pilot.groupby('dominant_axis')['agree'].agg(['mean', 'count']))
print(f"\noverall sign agreement: {pilot['agree'].mean():.1%} "
      f"over {len(pilot)} predictions ({pilot['pair_id'].nunique()} pairs)")

                   mean  count
dominant_axis                 
x              0.693548     62
y              0.662791     86
z              0.900000     10

overall sign agreement: 69.0% over 158 predictions (79 pairs)


## 6. Export `pairs.json`

Written to Drive, then transferred to the rendering instance, e.g.:

```
scp -i key.pem pairs.json ubuntu@<instance-ip>:~/
```

The visualiser is run on the instance with Isaac Sim's bundled interpreter:
`./python.sh visualise_pairs.py --pairs ~/pairs.json --out ./figs`.


In [13]:
write_pairs(pairs, OUT_JSON)

[write_pairs] wrote 79 pairs -> /content/drive/MyDrive/openvla_cache/pairs.json


'/content/drive/MyDrive/openvla_cache/pairs.json'

## 7. Preview a pair

Spot check of one exported record: instructions, mean action vectors, and the
x-axis sign relationship that the left/right probes target.


In [14]:
import numpy as np
p = pairs[0]
print('pair_id :', p['pair_id'], '| scene:', p['scene_id'])
print('A:', p['instr_a'])
print('   action =', np.round(p['action_a'], 4), f"(n={p['n_a']})")
print('B:', p['instr_b'])
print('   action =', np.round(p['action_b'], 4), f"(n={p['n_b']})")
dx_a, dx_b = p['action_a'][0], p['action_b'][0]
print(f'dx(A) = {dx_a:+.4f}   dx(B) = {dx_b:+.4f}   '
      f"x-sign flip: {'YES' if dx_a * dx_b < 0 else 'no'}")
if 'gt_vector' in p:
    print('gt      =', np.round(p['gt_vector'], 4))

pair_id : ep000000_left | scene: 0
A: Place the can to the left of the pot.
   action = [-2.700e-03  5.000e-04 -8.000e-03  1.100e-03  2.530e-02  8.000e-03
  9.961e-01] (n=1)
B: Place the can to the right of the pot.
   action = [-2.700e-03  5.000e-04 -8.800e-03  1.100e-03  3.000e-02 -9.700e-03
  9.961e-01] (n=1)
dx(A) = -0.0027   dx(B) = -0.0027   x-sign flip: no
gt      = [-0.0868  0.0915 -0.0258 -0.0594 -0.035   1.4381  1.    ]
